# Sherdog ufc web crawler and analysis

In [ ]:
import pandas as pd
import numpy as np

fights = pd.read_csv("../data/fights.csv")
fighters = pd.read_csv("../data/fighters.csv")
events = pd.read_csv("../data/events.csv")

print(fights.shape, fighters.shape, events.shape)


In [ ]:
# --- Section 4: what the crawl actually captured, over time ---
# A genuine distribution: every scraped fight is counted exactly once, and the
# bars sum to the full corpus. This is what the report's "data scraped" section
# needs, and it makes the dataset's temporality (lecture slide 6) visible:
# the UFC's own growth means recent eras dominate any pooled statistic.
import matplotlib.pyplot as plt

years = pd.to_datetime(fights["event_date"]).dt.year
per_year = years.value_counts().sort_index()

fig, ax = plt.subplots(figsize=(9, 4.4), facecolor="#fcfcfb")
ax.set_facecolor("#fcfcfb")

bars = ax.bar(per_year.index, per_year.values, color="#2a78d6", width=0.75, zorder=3)
bars[-1].set_color("#898781")  # 2026 is a partial year, greyed so it isn't read as a decline

# headroom so the annotation sits in clear air above the bars instead of on them
ax.set_ylim(0, 850)
ax.annotate("2026 is a partial year\n(data to August)",
            xy=(per_year.index[-1], per_year.values[-1] + 15),
            xytext=(per_year.index[-1] - 7.5, 800),
            fontsize=11, color="#52514e", ha="center", va="top",
            arrowprops=dict(arrowstyle="->", color="#898781", linewidth=1))

ax.set_ylabel("Fights scraped", color="#3d3c39", fontsize=13)
ax.set_xlabel("Year", color="#3d3c39", fontsize=13)

ax.spines[["top", "right"]].set_visible(False)
ax.spines[["left", "bottom"]].set_color("#898781")
ax.tick_params(colors="#3d3c39", labelsize=12)
ax.yaxis.grid(True, color="#d8d7cf", linewidth=0.8, zorder=0)
ax.set_axisbelow(True)

plt.tight_layout()
plt.savefig("../report/figures/fights_per_year.png", dpi=200, bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.show()


In [ ]:
# move fights from wide to long format
# this will allow us to do some nice chronological trend analysis
a = fights.rename(columns={"fighter_a_id": "fighter_id", "fighter_a_name": "fighter_name",
                           "fighter_b_id": "opponent_id", "fighter_b_name": "opponent_name"})
b = fights.rename(columns={"fighter_b_id": "fighter_id", "fighter_b_name": "fighter_name",
                           "fighter_a_id": "opponent_id", "fighter_a_name": "opponent_name"})
long_df = pd.concat([a, b], ignore_index=True)

print(long_df.shape)

long_df[long_df["fighter_id"] == 50529].sort_values("event_date")[
    ["fighter_id", "event_date", "fighter_name", "opponent_name", 'winner_id']
]

In [ ]:
# check for no contests
# right now if there is no context the winner id is NaN
def get_result(row):
    if row["outcome_type"] != "win":
        return row["outcome_type"]  # draw / nc / unknown
    
    return "win" if row["winner_id"] == row["fighter_id"] else "loss"

long_df["result"] = long_df.apply(get_result, axis=1)

long_df[long_df["fighter_id"] == 50529].sort_values("event_date")[
    ["event_date", "opponent_name", "result"]
]


In [ ]:
# leakage safe days since last fight
long_df["event_date"] = pd.to_datetime(long_df["event_date"]) # turn the string date into panda datetime
long_df = long_df.sort_values(["fighter_id", "event_date"]).reset_index(drop=True) # srot by the fighter id and event date

# clever part. diff calc rows val minus the previous row.
long_df["days_since_prior"] = long_df.groupby("fighter_id")["event_date"].diff().dt.days

long_df[long_df["fighter_id"] == 50529][["event_date", "opponent_name", "days_since_prior"]].head(6)


In [ ]:
# a fighters win rate computed only from previous fights
long_df["is_win"] = (long_df["result"] == "win").astype(int)

long_df["win_rate_entering"] = (long_df.groupby("fighter_id")["is_win"]
                       .apply(lambda s: s.shift().expanding().mean())
                       .reset_index(drop=True))

long_df[long_df["fighter_id"] == 50529][["event_date", "opponent_name", "result", "win_rate_entering"]].head(8)


In [ ]:
# we would also like the age of a fighter at the time of one of his fights
fighters["birth_date"] = pd.to_datetime(fighters["birth_date"])

# keep every row of df, attatch birth date where a matching fighter id exists in fighters
long_df = long_df.merge(fighters[["fighter_id", "birth_date"]], on="fighter_id", how="left")

long_df["age_at_fight"] = (long_df["event_date"] - long_df["birth_date"]).dt.days / 365.25

long_df[long_df["fighter_id"] == 50529][["event_date", "opponent_name", "age_at_fight"]].head(4)

In [ ]:
FINISH_METHODS = {"KO", "TKO", "Submission", "Technical Submission"}
long_df["is_finish"] = long_df["method_category"].isin(FINISH_METHODS).astype(int)

# group by fighter id
# for s (every fighters results 1, 0, 0, 1 etc) apply lambda function
# move each val one step back
# builds cum window from start fighter history to this point
# mean averages all fights. on avg how many of this fighters fight are a finish
long_df["finish_rate_entering"] = (
    long_df.groupby("fighter_id")["is_finish"]
    .apply(lambda s: s.shift().expanding().mean())
    .reset_index(drop=True)
)

long_df[long_df["fighter_id"] == 50529][
    ["event_date", "opponent_name", "method_category", "is_finish", "finish_rate_entering"]
].head(6)

In [ ]:
# merge the entering features we calculated into the wide fights.csv
# for each fight we want to be able to see both fighters pre fight stats
features = long_df[[
    "fight_id", "fighter_id", "win_rate_entering", "finish_rate_entering",
    "days_since_prior", "age_at_fight",
]]

a_features = features.rename(columns={
    "fighter_id": "fighter_a_id",
    "win_rate_entering": "a_win_rate_entering",
    "finish_rate_entering": "a_finish_rate_entering",
    "days_since_prior": "a_days_since_prior",
    "age_at_fight": "a_age_at_fight",
})
b_features = features.rename(columns={
    "fighter_id": "fighter_b_id",
    "win_rate_entering": "b_win_rate_entering",
    "finish_rate_entering": "b_finish_rate_entering",
    "days_since_prior": "b_days_since_prior",
    "age_at_fight": "b_age_at_fight",
})

fights_model = fights.merge(a_features, on=["fight_id", "fighter_a_id"], how="left")
fights_model = fights_model.merge(b_features, on=["fight_id", "fighter_b_id"], how="left")

fights_model[fights_model["fight_id"] == "101617-12"][
    ["event_date", "fighter_a_name", "fighter_b_name", "b_win_rate_entering", "b_finish_rate_entering"]
]

In [ ]:
# layoff length vs next fight performance

q2 = long_df.dropna(subset=["days_since_prior"]).copy()
q2 = q2[q2["result"].isin(["win", "loss"])]  # drop draws/nc — not meaningful for a win-rate comparison

q2["layoff_bucket"] = pd.cut(
    q2["days_since_prior"],
    bins=[0, 90, 180, 365, 730, np.inf],
    labels=["<90d", "90-180d", "180-365d", "1-2yr", "2yr+"],
)

layoff_winrate = q2.groupby("layoff_bucket", observed=True)["is_win"].agg(["mean", "count"])
print(layoff_winrate)


In [ ]:
# layoff length vs next fight performance
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8.5, 4.4), facecolor="#fcfcfb")
ax.set_facecolor("#fcfcfb")

buckets = layoff_winrate.index.tolist()
win_rates = layoff_winrate["mean"].values
counts = layoff_winrate["count"].values

bars = ax.bar(buckets, win_rates, color="#2a78d6", width=0.6, zorder=3)

ax.axhline(0.5, color="#52514e", linestyle="--", linewidth=1, zorder=2)
ax.text(0.99, 0.515, "50% = no advantage", transform=ax.get_yaxis_transform(),
        color="#52514e", fontsize=11.5, va="bottom", ha="right")

for bar, rate, n in zip(bars, win_rates, counts):
    ax.text(bar.get_x() + bar.get_width() / 2, rate + 0.012, f"{rate:.0%}",
            ha="center", va="bottom", fontsize=14.5, color="#0b0b0b")
    # white on the blue fill: the muted grey used elsewhere is unreadable here
    ax.text(bar.get_x() + bar.get_width() / 2, 0.02, f"n={n:,}",
            ha="center", va="bottom", fontsize=11.5, color="#ffffff")

ax.set_ylim(0, max(win_rates) + 0.1)
ax.set_ylabel("Win rate", color="#3d3c39", fontsize=13)
ax.set_xlabel("Days since previous fight", color="#3d3c39", fontsize=13)

ax.spines[["top", "right"]].set_visible(False)
ax.spines[["left", "bottom"]].set_color("#898781")
ax.tick_params(colors="#3d3c39", labelsize=12)
ax.yaxis.grid(True, color="#d8d7cf", linewidth=0.8, zorder=0)
ax.set_axisbelow(True)

plt.tight_layout()
plt.savefig("../report/figures/layoff_winrate.png", dpi=200, bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.show()


In [ ]:
backtest = fights_model.dropna(subset=["a_win_rate_entering", "b_win_rate_entering"])
backtest = backtest[backtest["outcome_type"] == "win"]

print(f"{len(backtest)} of {len(fights_model)} fights have enough history to backtest")

backtest = backtest[backtest["a_win_rate_entering"] != backtest["b_win_rate_entering"]]

predicted_a_wins = backtest["a_win_rate_entering"] > backtest["b_win_rate_entering"]
actual_a_wins = backtest["winner_id"] == backtest["fighter_a_id"]

backtest["correct"] = predicted_a_wins == actual_a_wins

accuracy = backtest["correct"].mean()
print(f"rule accuracy: {accuracy:.1%} over {len(backtest)} fights")
print(f"baseline (coin flip): 50.0%")


In [ ]:
# is ther a home country advantage
long_df = long_df.merge(events[["event_id", "location_raw"]], on="event_id", how="left")
long_df = long_df.merge(fighters[["fighter_id", "nationality"]], on="fighter_id", how="left")

long_df["event_country"] = long_df["location_raw"].str.split(",").str[-1].str.strip()
long_df["fighting_at_home"] = long_df["nationality"] == long_df["event_country"]

q5 = long_df.dropna(subset=["nationality", "event_country"])
q5 = q5[q5["result"].isin(["win", "loss"])]

home_winrate = q5.groupby("fighting_at_home")["is_win"].agg(["mean", "count"])
print(home_winrate)


In [ ]:
# is there a home fight advantage
fig, ax = plt.subplots(figsize=(7.5, 4.2), facecolor="#fcfcfb")
ax.set_facecolor("#fcfcfb")

labels = ["Away", "Home"]  # matches groupby's False, True order
win_rates = home_winrate["mean"].values
counts = home_winrate["count"].values

bars = ax.bar(labels, win_rates, color="#2a78d6", width=0.5, zorder=3)

ax.axhline(0.5, color="#52514e", linestyle="--", linewidth=1, zorder=2)
# sits well above the line: the 50.5% value label would otherwise collide with it
ax.text(0.99, 0.575, "50% = no advantage", transform=ax.get_yaxis_transform(),
        color="#52514e", fontsize=11.5, va="bottom", ha="right")

for bar, rate, n in zip(bars, win_rates, counts):
    ax.text(bar.get_x() + bar.get_width() / 2, rate + 0.012, f"{rate:.1%}",
            ha="center", va="bottom", fontsize=15.5, color="#0b0b0b")
    ax.text(bar.get_x() + bar.get_width() / 2, 0.02, f"n={n:,}",
            ha="center", va="bottom", fontsize=11.5, color="#ffffff")

ax.set_ylim(0, 0.65)
ax.set_ylabel("Win rate", color="#3d3c39", fontsize=13)

ax.spines[["top", "right"]].set_visible(False)
ax.spines[["left", "bottom"]].set_color("#898781")
ax.tick_params(colors="#3d3c39", labelsize=12)
ax.yaxis.grid(True, color="#d8d7cf", linewidth=0.8, zorder=0)
ax.set_axisbelow(True)

plt.tight_layout()
plt.savefig("../report/figures/home_advantage.png", dpi=200, bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.show()


In [ ]:
# age when do fighters peak -- distribution first, so the Q3 bin edges below
# are chosen against the real shape of the data rather than guessed.
print(long_df["age_at_fight"].describe())

fig, ax = plt.subplots(figsize=(8.5, 4.3), facecolor="#fcfcfb")
ax.set_facecolor("#fcfcfb")

ages = long_df["age_at_fight"].dropna()
counts, _, _ = ax.hist(ages, bins=50, color="#2a78d6", zorder=3)

# headroom above the tallest bar so the median label sits in clear air:
# the median lands almost exactly on the peak, so there is no room beside it
ax.set_ylim(0, counts.max() * 1.2)
ax.set_xlim(18, 48)  # data stops at ~46; the default ran to 60 and left the panel a third empty

median_age = ages.median()
ax.axvline(median_age, color="#52514e", linestyle="--", linewidth=1, zorder=4)
ax.text(median_age + 0.5, 0.96, f"median {median_age:.1f} yrs",
        transform=ax.get_xaxis_transform(), color="#52514e", fontsize=11.5, va="top")

ax.set_ylabel("Fights", color="#3d3c39", fontsize=13)
ax.set_xlabel("Age at time of fight (years)", color="#3d3c39", fontsize=13)

ax.spines[["top", "right"]].set_visible(False)
ax.spines[["left", "bottom"]].set_color("#898781")
ax.tick_params(colors="#3d3c39", labelsize=12)
ax.yaxis.grid(True, color="#d8d7cf", linewidth=0.8, zorder=0)
ax.set_axisbelow(True)

plt.tight_layout()
plt.savefig("../report/figures/age_distribution.png", dpi=200, bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.show()


In [ ]:
# fiilter to win loss subset with age
q3 = long_df[
    long_df["result"].isin(["win", "loss"]) & long_df["age_at_fight"].notna()
].copy()

# bin age and calculate win rate per bin
bins = [21, 24, 27, 30, 33, 36, 39, 100]
labels = ["21-23", "24-26", "27-29", "30-32", "33-35", "36-38", "39+"]

q3["age_bin"] = pd.cut(q3["age_at_fight"], bins=bins, labels=labels, right=False)

age_winrate = q3.groupby("age_bin", observed=True)["result"].agg(
    mean=lambda s: (s == "win").mean(), count="count"
)
age_winrate




In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.4), facecolor="#fcfcfb")
ax.set_facecolor("#fcfcfb")

x = range(len(age_winrate))
y = age_winrate["mean"].values
n = age_winrate["count"].values

ax.plot(x, y, color="#2a78d6", marker="o", markersize=6, linewidth=2.2, zorder=3)

ax.axhline(0.5, color="#52514e", linestyle="--", linewidth=1, zorder=2)
ax.text(0.99, 0.515, "50% = no advantage", transform=ax.get_yaxis_transform(),
        color="#52514e", fontsize=11.5, va="bottom", ha="right")

for xi, yi, ni in zip(x, y, n):
    ax.text(xi, yi + 0.018, f"{yi:.1%}", ha="center", va="bottom",
             fontsize=13, color="#0b0b0b")
    ax.text(xi, 0.02, f"n={ni:,}", ha="center", va="bottom",
             fontsize=11, color="#52514e")

ax.set_xticks(list(x))
ax.set_xticklabels(age_winrate.index, color="#3d3c39", fontsize=12)
ax.set_ylim(0, 0.68)
ax.set_ylabel("Win rate", color="#3d3c39", fontsize=13)
ax.set_xlabel("Age at time of fight (years)", color="#3d3c39", fontsize=13)

ax.spines[["top", "right"]].set_visible(False)
ax.spines[["left", "bottom"]].set_color("#898781")
ax.tick_params(colors="#3d3c39", labelsize=12)
ax.yaxis.grid(True, color="#d8d7cf", linewidth=0.8, zorder=0)
ax.set_axisbelow(True)

plt.tight_layout()
plt.savefig("../report/figures/age_winrate.png", dpi=200, bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.show()


In [ ]:
# finish method over time
fights["event_date"] = pd.to_datetime(fights["event_date"])
fights["method_category"].value_counts()

# check era coverage
fights["event_date"].dt.year.value_counts().sort_index()

In [ ]:
#KO and TKO one bucket
method_map = {
    "KO": "KO/TKO",
    "TKO": "KO/TKO",
    "Decision": "Decision",
    "Submission": "Submission",
}
fights["method_simple"] = fights["method_category"].map(method_map).fillna("Other")
fights["method_simple"].value_counts()

In [ ]:
# bin fights into eras
year = fights["event_date"].dt.year
era_bins = [1993, 2000, 2010, 2020, 2027]
era_labels = ["1993-1999", "2000-2009", "2010-2019", "2020-2026"]
fights["era"] = pd.cut(year, bins=era_bins, labels=era_labels, right=False)

finishes = fights[fights["method_simple"] != "Other"]
method_by_era = pd.crosstab(finishes["era"], finishes["method_simple"], normalize="index")
method_by_era


In [ ]:
# redesigned: line chart instead of a 100%-stacked bar. A stacked bar makes the
# middle segment (KO/TKO) sit on a shifting baseline across eras, which the
# lecture specifically flags as hard to compare (slide 101-102). A line per
# method reads its value directly off the shared y-axis at every era instead.
categories = ["Decision", "KO/TKO", "Submission"]
colors = {"Decision": "#2a78d6", "KO/TKO": "#eb6834", "Submission": "#1baf7a"}

era_n = finishes.groupby("era", observed=True).size()

fig, ax = plt.subplots(figsize=(9, 4.6), facecolor="#fcfcfb")
ax.set_facecolor("#fcfcfb")

x = list(range(len(method_by_era.index)))
for cat in categories:
    y = method_by_era[cat].values
    ax.plot(x, y, color=colors[cat], marker="o", markersize=6, linewidth=2.2, zorder=3)
    ax.text(x[-1] + 0.12, y[-1], cat, color=colors[cat], fontsize=13,
            va="center", fontweight="bold")

# Sample size per era. Everything here is a *share*, which hides the fact that
# the 1990s bin rests on 225 fights and the 2010s on 4,033: an 18x difference.
# Without this the earliest point looks as solid as the latest one; it isn't.
for xi, era in zip(x, method_by_era.index):
    ax.text(xi, 0.015, f"n={era_n[era]:,}", ha="center", va="bottom",
            fontsize=11, color="#52514e")

ax.set_xlim(-0.3, 4.15)
ax.set_xticks(x)
ax.set_xticklabels(method_by_era.index, color="#3d3c39", fontsize=12)
ax.set_ylim(0, 0.6)
ax.set_ylabel("Share of fights", color="#3d3c39", fontsize=13)
ax.set_xlabel("Era", color="#3d3c39", fontsize=13)

ax.spines[["top", "right"]].set_visible(False)
ax.spines[["left", "bottom"]].set_color("#898781")
ax.tick_params(colors="#3d3c39", labelsize=12)
ax.yaxis.grid(True, color="#d8d7cf", linewidth=0.8, zorder=0)
ax.set_axisbelow(True)

plt.tight_layout()
plt.savefig("../report/figures/method_by_era.png", dpi=200, bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.show()


In [ ]:
# how long does a finish take, split by KO/TKO vs Submission? Decision is left
# out on purpose -- it sits at exactly 15 or 25 minutes by definition of the
# round clock, which is a rule fact, not something the data found.
dur = finishes[finishes["end_round"] > 0].copy()  # one UFC 1 bout has end_round 0, no round ever recorded

def to_minutes(row):
    m, s = row["end_time"].split(":")
    return (row["end_round"] - 1) * 5 + int(m) + int(s) / 60

dur["total_min"] = dur.apply(to_minutes, axis=1)
finish_categories = ["KO/TKO", "Submission"]
groups = [dur.loc[dur["method_simple"] == c, "total_min"].values for c in finish_categories]

fig, ax = plt.subplots(figsize=(6, 5), facecolor="#fcfcfb")
ax.set_facecolor("#fcfcfb")
positions = range(1, len(finish_categories) + 1)
parts = ax.violinplot(groups, positions=positions, showmedians=True, showextrema=False, widths=0.75)
for body, cat in zip(parts["bodies"], finish_categories):
    body.set_facecolor(colors[cat])
    body.set_edgecolor(colors[cat])
    body.set_alpha(0.55)
    body.set_zorder(3)
parts["cmedians"].set_color("#0b0b0b")
parts["cmedians"].set_linewidth(1.6)

for x, g in zip(positions, groups):
    med = pd.Series(g).median()
    ax.text(x, med + 0.6, f"{med:.1f}", ha="center", va="bottom",
            fontsize=12, color="#0b0b0b", fontweight="bold", zorder=5)
    ax.text(x, -1.8, f"n={len(g):,}", ha="center", va="top", fontsize=10, color="#898781")

ax.set_xticks(list(positions))
ax.set_xticklabels(finish_categories, color="#3d3c39", fontsize=13)
ax.set_ylabel("Total fight time (minutes)", color="#3d3c39", fontsize=13)
ax.set_ylim(-3, 27)
ax.spines[["top", "right"]].set_visible(False)
ax.spines[["left", "bottom"]].set_color("#898781")
ax.tick_params(colors="#3d3c39", labelsize=11)
ax.yaxis.grid(True, color="#d8d7cf", linewidth=0.8, zorder=0)
ax.set_axisbelow(True)

plt.tight_layout()
plt.savefig("../report/figures/duration_by_method.png", dpi=200, bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.show()

In [ ]:
# how have specfic sibmission types changed over years
subs = fights[fights["method_simple"] == "Submission"]
subs["method_subtype"].value_counts()


In [ ]:
subs["method_subtype"].value_counts().to_string()


In [ ]:
# formalise format mistakes e.g. "rear naked choke" rear-naked choke
subs = subs.copy()
subs["subtype_clean"] = subs["method_subtype"].replace({
    "Rear Naked Choke": "Rear-Naked Choke",
    "Arm Bar": "Armbar",
})

# top popular submissions by era
top_subs = [
    "Rear-Naked Choke", "Guillotine Choke", "Armbar", "Arm-Triangle Choke",
    "Triangle Choke", "Kimura", "Brabo Choke",
]
subs["sub_simple"] = subs["subtype_clean"].where(subs["subtype_clean"].isin(top_subs), "Other")

subs_top = subs[subs["sub_simple"] != "Other"]
sub_by_era = pd.crosstab(subs_top["era"], subs_top["sub_simple"], normalize="index")
sub_by_era = sub_by_era[top_subs]
sub_by_era




In [ ]:
# redesigned: small multiples (one mini line-trend panel per submission type)
# instead of a 7-segment stacked bar. Same baseline-comparison problem as Q1,
# made worse by 7 segments, and the lecture's own fix for "large class
# counts" (slide 109) plus "juxtaposition / small multiples" (slide 107) is
# exactly this: separate panels, same y-scale, laid out side by side.
sub_n = subs_top.groupby("era", observed=True).size()

fig, axes = plt.subplots(2, 4, figsize=(11, 6.6), facecolor="#fcfcfb", sharey=True)
axes = axes.flatten()

era_labels_short = ["'93-99", "'00s", "'10s", "'20s"]
x = list(range(len(sub_by_era.index)))

for i, cat in enumerate(top_subs):
    ax = axes[i]
    ax.set_facecolor("#fcfcfb")
    y = sub_by_era[cat].values
    ax.plot(x, y, color="#2a78d6", marker="o", markersize=4, linewidth=2, zorder=3)
    ax.set_title(cat, fontsize=14.5, color="#0b0b0b", loc="left")
    ax.set_xticks(x)
    ax.set_xticklabels(era_labels_short, color="#3d3c39", fontsize=12)
    ax.set_ylim(0, 0.55)
    ax.spines[["top", "right"]].set_visible(False)
    ax.spines[["left", "bottom"]].set_color("#898781")
    ax.tick_params(colors="#3d3c39", labelsize=10.5)
    ax.yaxis.grid(True, color="#d8d7cf", linewidth=0.8, zorder=0)
    ax.set_axisbelow(True)

# The 8th grid slot is unused (7 categories), so spend it on the sample sizes
# rather than leaving it blank. The '93-99 column rests on 45 submissions, so
# every point above it moves by ~2pp per single fight; the '10s column has 609.
axes[-1].axis("off")
note = "Submissions per era\n\n" + "\n".join(
    f"{lab}    n = {sub_n[era]:,}" for lab, era in zip(era_labels_short, sub_by_era.index)
)
axes[-1].text(0.0, 0.82, note, fontsize=11.5, color="#52514e",
              va="top", ha="left", transform=axes[-1].transAxes, linespacing=1.7)

fig.supylabel("Share of common submission types", fontsize=12.5, color="#3d3c39", x=0.005)

plt.tight_layout()
plt.savefig("../report/figures/submission_taxonomy.png", dpi=200, bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.show()


In [ ]:
# who are the most enteratining fighters and the most boring!
# signals like finish rate, finish speed and fight activity
long_df["event_date"].max()
cutoff = long_df["event_date"].max() - pd.DateOffset(months=18)
recent_fighters = long_df[long_df["event_date"] >= cutoff]["fighter_id"].value_counts()
recent_fighters.describe()

In [ ]:
# who qualifies.-> uses recent window 18 months
# finish rate and finish speed are computed over the entire career
career = long_df.groupby("fighter_id").agg(
    finish_rate=("is_finish", "mean"), total_fights=("is_finish", "count"),
)
career

In [ ]:
# average finish round, for finishes only, filter is_finish
finish_speed = (
    long_df[long_df["is_finish"] == 1]
    .assign(end_round=lambda d: pd.to_numeric(d["end_round"], errors="coerce"))
    .groupby("fighter_id")["end_round"].mean()
)

eligible_ids = recent_fighters[recent_fighters >= 3].index

fighter_score = career.join(finish_speed.rename("finish_speed")).join(
    recent_fighters.rename("activity")
)
fighter_score = fighter_score.loc[fighter_score.index.intersection(eligible_ids)]


In [ ]:
# fighter number who have never finished or been finished
fighter_score["finish_speed"].isna().sum()

In [ ]:
long_df["end_round"] = pd.to_numeric(long_df["end_round"], errors="coerce")
long_df["end_round"].max()


In [ ]:
fighter_score["finish_speed"] = fighter_score["finish_speed"].fillna(long_df["end_round"].max())


In [ ]:
def minmax(s):
    return (s - s.min()) / (s.max() - s.min())

fighter_score["finish_rate_norm"] = minmax(fighter_score["finish_rate"])
fighter_score["finish_speed_norm"] = 1 - minmax(fighter_score["finish_speed"])  # invert: lower round = more exciting
fighter_score["activity_norm"] = minmax(fighter_score["activity"])

fighter_score["entertainment_score"] = fighter_score[
    ["finish_rate_norm", "finish_speed_norm", "activity_norm"]
].mean(axis=1)


In [ ]:
fighter_score = fighter_score.merge(
    fighters[["fighter_id", "name"]], left_index=True, right_on="fighter_id"
).set_index("fighter_id")

fighter_score.sort_values(["entertainment_score", "total_fights"], ascending=[True, False]).head(5)


In [ ]:
# simplest possible story: select AND display by the same metric (finish rate).
# tiebreak on total_fights so a 100% rate over many fights beats a 100% rate over just 3.
top5 = fighter_score.sort_values(["finish_rate", "total_fights"], ascending=[False, False]).head(5)
top5


In [ ]:
# every top-5-by-finish-rate candidate ties at 100%, so finish rate has hit a
# ceiling here and a bar chart of it shows 5 identical bars and no information.
# The thing that actually varies among them, and the thing the tiebreak was
# already using, is how many fights that perfect record covers. Show that instead.
fig, ax = plt.subplots(figsize=(8, 4.2), facecolor="#fcfcfb")
ax.set_facecolor("#fcfcfb")

ranked = top5.sort_values("total_fights")  # ascending so the longest streak plots at the top

bars = ax.barh(ranked["name"], ranked["total_fights"], color="#2a78d6", height=0.6, zorder=3)

for bar, (_, row) in zip(bars, ranked.iterrows()):
    ax.text(bar.get_width() + 0.25, bar.get_y() + bar.get_height() / 2,
            f"{row['total_fights']:.0f}", va="center", fontsize=15.5, color="#0b0b0b")

ax.set_xlim(0, 16)
ax.set_xlabel("Career fights, every one of them finished (no decisions)", color="#3d3c39", fontsize=13)

ax.spines[["top", "right", "left"]].set_visible(False)
ax.spines["bottom"].set_color("#898781")
ax.tick_params(colors="#3d3c39", labelsize=12)
ax.xaxis.grid(True, color="#d8d7cf", linewidth=0.8, zorder=0)
ax.set_axisbelow(True)

plt.tight_layout()
plt.savefig("../report/figures/finish_streaks.png", dpi=200, bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.show()


In [ ]:
# build the simple predictor from signals
long_df["finish_rate_entering"].describe()



In [ ]:
(long_df["finish_rate_entering"] == 0).mean()


In [ ]:
# does finish rate entering have any effect on outcome result
# no
q_finish = long_df[
    long_df["finish_rate_entering"].notna() & long_df["result"].isin(["win", "loss"])
].copy()

q_finish["finish_bucket"] = pd.qcut(q_finish["finish_rate_entering"], 4)

finish_winrate = q_finish.groupby("finish_bucket", observed=True)["is_win"].agg(["mean", "count"])
finish_winrate


In [ ]:
[c for c in fights_model.columns if "age" in c or "days_since" in c or "win_rate" in c]


In [ ]:
# backtest and predict using 3 signals
# win rate entering higher +1
# age at fight lower +1
#days since prior lower +1

#use simple point score card
q6 = fights_model[
    (fights_model["outcome_type"] == "win") &
    fights_model["a_win_rate_entering"].notna() & fights_model["b_win_rate_entering"].notna() &
    fights_model["a_age_at_fight"].notna() & fights_model["b_age_at_fight"].notna() &
    fights_model["a_days_since_prior"].notna() & fights_model["b_days_since_prior"].notna()
].copy()
len(q6)


In [ ]:
q6["a_points"] = (
    (q6["a_win_rate_entering"] > q6["b_win_rate_entering"]).astype(int) +
    (q6["a_age_at_fight"] < q6["b_age_at_fight"]).astype(int) +
    (q6["a_days_since_prior"] < q6["b_days_since_prior"]).astype(int)
)
q6["b_points"] = (
    (q6["b_win_rate_entering"] > q6["a_win_rate_entering"]).astype(int) +
    (q6["b_age_at_fight"] < q6["a_age_at_fight"]).astype(int) +
    (q6["b_days_since_prior"] < q6["a_days_since_prior"]).astype(int)
)


In [ ]:
q6["predicted_winner"] = np.select(
    [q6["a_points"] > q6["b_points"], q6["b_points"] > q6["a_points"]],
    [q6["fighter_a_id"], q6["fighter_b_id"]],
    default=np.nan,  # fully tied on all 3 signals — no prediction
)
accuracy = (q6["predicted_winner"] == q6["winner_id"]).mean()
accuracy


In [ ]:
# just win rate as signal
q6["winrate_only_predicted"] = np.select(
    [q6["a_win_rate_entering"] > q6["b_win_rate_entering"],
     q6["b_win_rate_entering"] > q6["a_win_rate_entering"]],
    [q6["fighter_a_id"], q6["fighter_b_id"]],
    default=np.nan,
)

# score both the same way: drop undecidable rows, don't count them as misses
scorecard_scored = q6.dropna(subset=["predicted_winner"])
scorecard_accuracy = (scorecard_scored["predicted_winner"] == scorecard_scored["winner_id"]).mean()
print(f"scorecard: {scorecard_accuracy:.1%} over {len(scorecard_scored)} fights")

winrate_scored = q6.dropna(subset=["winrate_only_predicted"])
winrate_accuracy = (winrate_scored["winrate_only_predicted"] == winrate_scored["winner_id"]).mean()
print(f"win-rate only: {winrate_accuracy:.1%} over {len(winrate_scored)} fights")



In [ ]:
# The headline result.
labels = ["Win rate only", "Win rate + age + layoff"]
values = [winrate_accuracy, scorecard_accuracy]
ns = [len(winrate_scored), len(scorecard_scored)]

fig, ax = plt.subplots(figsize=(8, 4.3), facecolor="#fcfcfb")
ax.set_facecolor("#fcfcfb")

bars = ax.bar(labels, values, color="#2a78d6", width=0.4, zorder=3)

ax.axhline(0.5, color="#52514e", linestyle="--", linewidth=1, zorder=4)

ax.text(0.5, 0.508, "50% = coin flip", ha="center", va="bottom",
        fontsize=11.5, color="#52514e")

for bar, v, n in zip(bars, values, ns):
    ax.text(bar.get_x() + bar.get_width() / 2, v + 0.012, f"{v:.1%}",
            ha="center", va="bottom", fontsize=15, color="#0b0b0b")
    ax.text(bar.get_x() + bar.get_width() / 2, 0.02, f"n={n:,} fights",
            ha="center", va="bottom", fontsize=11.5, color="#ffffff")

ax.set_ylim(0, 0.7)
ax.set_ylabel("Correctly predicted winners", color="#3d3c39", fontsize=13)

ax.spines[["top", "right"]].set_visible(False)
ax.spines[["left", "bottom"]].set_color("#898781")
ax.tick_params(colors="#3d3c39", labelsize=12)
ax.yaxis.grid(True, color="#d8d7cf", linewidth=0.8, zorder=0)
ax.set_axisbelow(True)

plt.tight_layout()
plt.savefig("../report/figures/backtest.png", dpi=200, bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.show()


## Do bigger crowds see more finishes?

In [ ]:
attendance = pd.read_csv("../data/attendance.csv")
attendance["date"] = pd.to_datetime(attendance["date"])
attendance["date"].describe()
attendance["attendance"].notna().mean()


In [ ]:
# join on date, not name sherdog and wiki both share the same dae of event accordingly
events_dated = events.copy()
events_dated["date"] = pd.to_datetime(events_dated["date"])

# treat any date not unique from wiki as unreliable and drop it
att_dates = attendance["date"].value_counts()
ambiguous = att_dates[att_dates > 1].index
print(f"{len(ambiguous)} dates are ambiguous on the Wikipedia side and excluded from the join")
attendance_unique = attendance[~attendance["date"].isin(ambiguous)]

events_dated = events_dated.merge(attendance_unique[["date", "attendance"]], on="date", how="left")
assert len(events_dated) == len(events), "join must stay 1:1 with events.csv"

print(f"{events_dated['attendance'].notna().sum()} of {len(events_dated)} events have an attendance figure")

# a handful of dates hosted two separate UFC cards. like TUF and fight night
dupe_dates = events_dated["date"].value_counts()
print(f"{(dupe_dates > 1).sum()} dates hosted more than one UFC event")


In [ ]:
# attach attendance to each fight via its event, then bin into quartiles
fights_att = fights.merge(events_dated[["event_id", "attendance"]], on="event_id", how="left")
fights_att["is_finish"] = fights_att["method_category"].isin(
    ["KO", "TKO", "Submission", "Technical Submission"]
).astype(int)

q8 = fights_att.dropna(subset=["attendance"]).copy()
print(f"{len(q8)} of {len(fights_att)} fights have a joined attendance figure")

q8["attendance_bin"] = pd.qcut(q8["attendance"], 4)
attendance_finish = q8.groupby("attendance_bin", observed=True)["is_finish"].agg(["mean", "count"])
attendance_finish

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.4), facecolor="#fcfcfb")
ax.set_facecolor("#fcfcfb")

x = range(len(attendance_finish))
y = attendance_finish["mean"].values
n = attendance_finish["count"].values
labels = [f"{int(iv.left/1000)}k-{int(iv.right/1000)}k" for iv in attendance_finish.index]

ax.plot(x, y, color="#2a78d6", marker="o", markersize=6, linewidth=2.2, zorder=3)

ax.axhline(0.5, color="#52514e", linestyle="--", linewidth=1, zorder=2)

ax.text(1.5, 0.28, "50% baseline finish rate", color="#52514e", fontsize=11, ha="center")

for xi, yi, ni in zip(x, y, n):
    ax.text(xi, yi + 0.018, f"{yi:.1%}", ha="center", va="bottom", fontsize=13, color="#0b0b0b")
    ax.text(xi, 0.02, f"n={ni:,}", ha="center", va="bottom", fontsize=9.5, color="#52514e")

ax.set_xticks(list(x))
ax.set_xticklabels(labels, color="#3d3c39", fontsize=12)
ax.set_ylim(0, 0.65)
ax.set_ylabel("Finish rate", color="#3d3c39", fontsize=13)
ax.set_xlabel("Event attendance (quartile)", color="#3d3c39", fontsize=13)

ax.spines[["top", "right"]].set_visible(False)
ax.spines[["left", "bottom"]].set_color("#898781")
ax.tick_params(colors="#3d3c39", labelsize=12)
ax.yaxis.grid(True, color="#d8d7cf", linewidth=0.8, zorder=0)
ax.set_axisbelow(True)

plt.tight_layout()
plt.savefig("../report/figures/attendance_finish_rate.png", dpi=200, bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.show()

## Worked example of the image pipeline


In [ ]:
import os
from PIL import Image

IMAGES_STORE = "../data/images"

def career_stats(fighter_id):
    rows = fights[(fights["fighter_a_id"] == fighter_id) | (fights["fighter_b_id"] == fighter_id)]
    wins = (rows["winner_id"] == fighter_id).sum()
    total = len(rows)
    finishes = rows["method_category"].isin(
        ["KO", "TKO", "Submission", "Technical Submission"]
    ).sum()
    titles = rows["is_title_fight"].sum()
    return wins, total - wins, finishes, titles

profiles = [
    (146193, "Dricus du Plessis", "South Africa"),
    (232591, "Terrance McKinney", "United States"),
]

fig, axes = plt.subplots(1, 2, figsize=(9, 5), facecolor="#fcfcfb")

for ax, (fid, name, nat) in zip(axes, profiles):
    row = fighters[fighters["fighter_id"] == fid].iloc[0]
    img = Image.open(os.path.join(IMAGES_STORE, row["image_path"]))
    ax.imshow(img)
    ax.axis("off")

    wins, losses, finishes, titles = career_stats(fid)
    finish_rate = finishes / (wins + losses)
    stats = (
        f"{name}\n{nat}\n\n"
        f"Record: {wins}-{losses}\n"
        f"Finish rate: {finish_rate:.0%}\n"
        f"Title fights: {titles}\n"
        f"Portrait: {int(row['image_width'])}\u00d7{int(row['image_height'])}px, "
        f"{row['image_kb']:.0f} KB"
    )
    ax.text(0.5, -0.06, stats, transform=ax.transAxes, ha="center", va="top",
            fontsize=11, color="#0b0b0b", linespacing=1.6)

plt.tight_layout(rect=[0, 0.06, 1, 1])
plt.savefig("../report/figures/fighter_portraits.png", dpi=200, bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.show()
